# Laboratorio 2: Deep Learning aplicado a series de tiempo

**Curso:** CC3084 — Data Science  
**Fecha:** julio 2026

## Avance

En este avance se construyen y comparan modelos Long Short-Term Memory (LSTM) para dos series temporales del laboratorio anterior:

1. **Total mensual** (Turista + Excursionista)
2. **Frontera 01 La Aurora** (mayor volumen acumulado)

Para cada serie se evalúan cuatro configuraciones de arquitectura e hiperparámetros, conservando los mismos conjuntos de entrenamiento y prueba (`outputs/parte1/series/`).

> **Nota de entorno:** TensorFlow no publica ruedas para Python 3.14. Se usa **Keras 3 con backend Torch**, con la misma API (`Sequential`, `LSTM`, `EarlyStopping`) del enunciado.


## Instalación de dependencias

Ejecute solo si faltan paquetes. Luego reinicie el kernel si es necesario.


In [ ]:
%pip install -q keras torch pandas numpy matplotlib scikit-learn


## Importación de librerías

Se establecen semillas para reducir la variación entre ejecuciones. Esto permite que el procedimiento sea más reproducible.


In [ ]:
from pathlib import Path
import sys
import os

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.environ.setdefault("KERAS_BACKEND", "torch")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mpl-config")

from src import lstm_lab as L
import pandas as pd
import matplotlib.pyplot as plt

print("ROOT:", ROOT)
print("Keras backend:", __import__("keras").backend.backend())
print("Series dir:", L.SERIES_DIR)


## Carga de las dos series

Se reutilizan exactamente los CSV `train`/`test` del laboratorio anterior. No se hace una división aleatoria.


In [ ]:
train_serie_1, test_serie_1 = L.cargar_serie(L.SERIE_1_SLUG)
train_serie_2, test_serie_2 = L.cargar_serie(L.SERIE_2_SLUG)

NOMBRE_SERIE_1 = L.NOMBRE_SERIE_1
NOMBRE_SERIE_2 = L.NOMBRE_SERIE_2

train_serie_1 = L.convertir_a_serie(train_serie_1, "train_serie_1")
test_serie_1 = L.convertir_a_serie(test_serie_1, "test_serie_1")
train_serie_2 = L.convertir_a_serie(train_serie_2, "train_serie_2")
test_serie_2 = L.convertir_a_serie(test_serie_2, "test_serie_2")

resumen_datos = L.resumen_conjuntos(
    train_serie_1, test_serie_1, train_serie_2, test_serie_2
)
resumen_datos


Los conjuntos anteriores son los mismos utilizados en el laboratorio anterior. El conjunto de prueba se mantiene separado durante el tuneo para evitar utilizar información futura al seleccionar los hiperparámetros.


## Visualización de las series


In [ ]:
def graficar_train_test_nb(train, test, nombre):
    plt.figure(figsize=(13, 5))
    plt.plot(train.index, train.values, label="Entrenamiento")
    plt.plot(test.index, test.values, label="Prueba")
    plt.title(f"División temporal — {nombre}")
    plt.xlabel("Tiempo")
    plt.ylabel("Valor de la serie")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

graficar_train_test_nb(train_serie_1, test_serie_1, NOMBRE_SERIE_1)
graficar_train_test_nb(train_serie_2, test_serie_2, NOMBRE_SERIE_2)


## Preparación de secuencias, métricas y modelos LSTM

Una red LSTM recibe tensores `muestras × pasos de tiempo × variables`. El parámetro `look_back` indica cuántas observaciones pasadas se usan para predecir la siguiente.

Métricas: **MAE**, **RMSE**, **MAPE** (omite ceros) y **R²**.

Las funciones viven en `src/lstm_lab.py` (`crear_secuencias`, `preparar_datos`, `construir_modelo_lstm`, `tunear_serie`, etc.).


In [ ]:
CONFIGURACIONES = L.CONFIGURACIONES
pd.DataFrame(CONFIGURACIONES)


Las configuraciones 1 y 2 cumplen el mínimo de dos modelos distintos. Las 3 y 4 amplían el tuneo (una vs dos capas LSTM). El tuneo usa validación temporal del final del entrenamiento; la prueba no participa en la selección. `EarlyStopping` recupera los mejores pesos.


## Tuneo para la primera serie


In [ ]:
resultados_serie_1, historiales_serie_1 = L.tunear_serie(
    train=train_serie_1,
    test=test_serie_1,
    nombre_serie=NOMBRE_SERIE_1,
    configuraciones=CONFIGURACIONES,
)
resultados_serie_1


### Resultados del tuneo de la primera serie

Se evaluaron **4** configuraciones. La de menor RMSE de validación fue **LSTM_1** (RMSE ≈ **110 769**, MAE ≈ **75 505**), con `look_back=6`, 32 unidades, dropout 0.1 y learning rate 0.001.


## Tuneo para la segunda serie


In [ ]:
resultados_serie_2, historiales_serie_2 = L.tunear_serie(
    train=train_serie_2,
    test=test_serie_2,
    nombre_serie=NOMBRE_SERIE_2,
    configuraciones=CONFIGURACIONES,
)
resultados_serie_2


### Resultados del tuneo de la segunda serie

Se evaluaron **4** configuraciones. La mejor fue **LSTM_1** (RMSE validación ≈ **43 493**, MAE ≈ **31 458**), también con una sola capa LSTM y `look_back=6`. Las redes más profundas/con mayor ventana no mejoraron la validación.


## Gráficas de pérdida durante el tuneo


In [ ]:
def graficar_historiales_nb(historiales, nombre_serie):
    plt.figure(figsize=(12, 6))
    for nombre_modelo, historial in historiales.items():
        plt.plot(historial["val_loss"], label=f"{nombre_modelo} - validación")
    plt.title(f"Pérdida de validación — {nombre_serie}")
    plt.xlabel("Época")
    plt.ylabel("MSE escalado")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

graficar_historiales_nb(historiales_serie_1, NOMBRE_SERIE_1)
graficar_historiales_nb(historiales_serie_2, NOMBRE_SERIE_2)


Las curvas de validación permiten observar la velocidad de convergencia y detectar sobreajuste. `EarlyStopping` evita continuar cuando la pérdida de validación deja de mejorar.


## Reentrenamiento y predicción con el mejor modelo

Después de seleccionar la mejor configuración con validación, se reentrena con todas las secuencias de entrenamiento usando la mejor época del tuneo, y se predice sobre prueba.


In [ ]:
mejor_config_serie_1 = resultados_serie_1.iloc[0]
modelo_final_serie_1, historial_final_serie_1, metricas_test_serie_1, pred_serie_1 = (
    L.entrenar_mejor_y_predecir(
        train=train_serie_1,
        test=test_serie_1,
        fila_mejor_configuracion=mejor_config_serie_1,
    )
)

print("Mejor configuración:")
display(mejor_config_serie_1.to_frame("Valor"))
print("Métricas sobre prueba:")
display(pd.DataFrame([metricas_test_serie_1], index=[NOMBRE_SERIE_1]))
pred_serie_1.head()


In [ ]:
mejor_config_serie_2 = resultados_serie_2.iloc[0]
modelo_final_serie_2, historial_final_serie_2, metricas_test_serie_2, pred_serie_2 = (
    L.entrenar_mejor_y_predecir(
        train=train_serie_2,
        test=test_serie_2,
        fila_mejor_configuracion=mejor_config_serie_2,
    )
)

print("Mejor configuración:")
display(mejor_config_serie_2.to_frame("Valor"))
print("Métricas sobre prueba:")
display(pd.DataFrame([metricas_test_serie_2], index=[NOMBRE_SERIE_2]))
pred_serie_2.head()


## Gráficas de valores reales y predicciones


In [ ]:
def graficar_predicciones_nb(predicciones, nombre_serie):
    plt.figure(figsize=(13, 5))
    plt.plot(predicciones.index, predicciones["Real"], marker="o", label="Valor real")
    plt.plot(predicciones.index, predicciones["Prediccion_LSTM"], marker="o", label="Predicción LSTM")
    plt.title(f"Valores reales y predicciones — {nombre_serie}")
    plt.xlabel("Tiempo")
    plt.ylabel("Valor")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

graficar_predicciones_nb(pred_serie_1, NOMBRE_SERIE_1)
graficar_predicciones_nb(pred_serie_2, NOMBRE_SERIE_2)


## Comparación de los mejores modelos


In [ ]:
comparacion_mejores = pd.DataFrame([
    {
        "Serie": NOMBRE_SERIE_1,
        "Mejor modelo": mejor_config_serie_1["modelo"],
        "Look back": int(mejor_config_serie_1["look_back"]),
        "MAE prueba": metricas_test_serie_1["MAE"],
        "RMSE prueba": metricas_test_serie_1["RMSE"],
        "MAPE prueba": metricas_test_serie_1["MAPE"],
        "R2 prueba": metricas_test_serie_1["R2"],
    },
    {
        "Serie": NOMBRE_SERIE_2,
        "Mejor modelo": mejor_config_serie_2["modelo"],
        "Look back": int(mejor_config_serie_2["look_back"]),
        "MAE prueba": metricas_test_serie_2["MAE"],
        "RMSE prueba": metricas_test_serie_2["RMSE"],
        "MAPE prueba": metricas_test_serie_2["MAPE"],
        "R2 prueba": metricas_test_serie_2["R2"],
    },
])
comparacion_mejores


## Comparación e interpretación

Para **Total mensual**, el mejor modelo fue **LSTM_1** (RMSE prueba ≈ **50 794**, MAE ≈ **40 554**, MAPE ≈ **18.05 %**).

Para **Frontera 01 La Aurora**, el mejor modelo fue **LSTM_1** (RMSE ≈ **19 295**, MAPE ≈ **18.82 %**).

En MAPE ambas series tienen error relativo similar; el total queda ligeramente mejor. La selección de hiperparámetros usó solo validación temporal.


## Comparación con los modelos del laboratorio anterior

Se contrastan contra los mejores modelos del informe (Holt-Winters multiplicativo para el total; ARIMA estacional para La Aurora).


In [ ]:
comparacion_laboratorios = pd.DataFrame([
    {
        "Serie": NOMBRE_SERIE_1,
        "Modelo": L.LAB1_MEJORES[NOMBRE_SERIE_1]["Modelo"],
        "MAE": L.LAB1_MEJORES[NOMBRE_SERIE_1]["MAE"],
        "RMSE": L.LAB1_MEJORES[NOMBRE_SERIE_1]["RMSE"],
        "MAPE": L.LAB1_MEJORES[NOMBRE_SERIE_1]["MAPE"],
    },
    {
        "Serie": NOMBRE_SERIE_1,
        "Modelo": "LSTM",
        "MAE": metricas_test_serie_1["MAE"],
        "RMSE": metricas_test_serie_1["RMSE"],
        "MAPE": metricas_test_serie_1["MAPE"],
    },
    {
        "Serie": NOMBRE_SERIE_2,
        "Modelo": L.LAB1_MEJORES[NOMBRE_SERIE_2]["Modelo"],
        "MAE": L.LAB1_MEJORES[NOMBRE_SERIE_2]["MAE"],
        "RMSE": L.LAB1_MEJORES[NOMBRE_SERIE_2]["RMSE"],
        "MAPE": L.LAB1_MEJORES[NOMBRE_SERIE_2]["MAPE"],
    },
    {
        "Serie": NOMBRE_SERIE_2,
        "Modelo": "LSTM",
        "MAE": metricas_test_serie_2["MAE"],
        "RMSE": metricas_test_serie_2["RMSE"],
        "MAPE": metricas_test_serie_2["MAPE"],
    },
])
comparacion_laboratorios


## Guardar resultados y modelos


In [ ]:
import json

L.RESULTADOS_DIR.mkdir(parents=True, exist_ok=True)
L.MODELOS_DIR.mkdir(parents=True, exist_ok=True)

resultados_serie_1.to_csv(L.RESULTADOS_DIR / "tuneo_serie_1.csv", index=False)
resultados_serie_2.to_csv(L.RESULTADOS_DIR / "tuneo_serie_2.csv", index=False)
comparacion_mejores.to_csv(L.RESULTADOS_DIR / "comparacion_mejores_lstm.csv", index=False)
comparacion_laboratorios.to_csv(L.RESULTADOS_DIR / "comparacion_laboratorio_anterior.csv", index=False)
pred_serie_1.to_csv(L.RESULTADOS_DIR / "predicciones_serie_1.csv")
pred_serie_2.to_csv(L.RESULTADOS_DIR / "predicciones_serie_2.csv")

modelo_final_serie_1.save(L.MODELOS_DIR / "mejor_lstm_serie_1.keras")
modelo_final_serie_2.save(L.MODELOS_DIR / "mejor_lstm_serie_2.keras")

with open(L.RESULTADOS_DIR / "configuracion_serie_1.json", "w", encoding="utf-8") as f:
    json.dump(mejor_config_serie_1.to_dict(), f, indent=4, default=str)
with open(L.RESULTADOS_DIR / "configuracion_serie_2.json", "w", encoding="utf-8") as f:
    json.dump(mejor_config_serie_2.to_dict(), f, indent=4, default=str)

texto = L.escribir_interpretaciones(
    resultados_serie_1,
    resultados_serie_2,
    mejor_config_serie_1,
    mejor_config_serie_2,
    metricas_test_serie_1,
    metricas_test_serie_2,
)
(L.RESULTADOS_DIR / "interpretaciones.md").write_text(texto, encoding="utf-8")
print("Resultados y modelos guardados correctamente.")
print(texto)


## Conclusiones

1. Se entrenaron **4** configuraciones LSTM por cada una de las dos series, variando `look_back`, unidades, capas, dropout, learning rate y batch size.
2. Mejor modelo para **Total mensual**: **LSTM_1** (RMSE prueba ≈ **50 794**).
3. Mejor modelo para **La Aurora**: **LSTM_1** (RMSE prueba ≈ **19 295**).
4. En MAPE, **Total mensual** (~18.1 %) fue ligeramente más preciso que La Aurora (~18.8 %).
5. Frente al laboratorio anterior, LSTM **superó** a Holt-Winters (total) y a ARIMA (La Aurora) en el mismo conjunto de prueba.

Detalle en `resultados/interpretaciones.md`.


## Atajo: regenerar todo desde cero

Si prefiere no entrenar celda por celda:

```bash
source .venv/bin/activate
export KERAS_BACKEND=torch
python -m src.lstm_lab
```
